In [1]:
import os

In [2]:
BASE_DIR = os.getcwd()

SOURCE_PATH = os.path.join(BASE_DIR, "data", "pokeData.csv")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")
TARGET_PATH = os.path.join(OUTPUT_DIR, "pokemon_target")
STAGING_PATH = os.path.join(OUTPUT_DIR, "pokemon_staging")
BACKUP_PATH = os.path.join(OUTPUT_DIR, "pokemon_backup")
WATERMARK_PATH = os.path.join(OUTPUT_DIR, "watermark.txt")

In [3]:
RUN_UNTIL = "2026-10-12 23:59:59"

In [4]:
from pyspark.sql import SparkSession

In [5]:
spark = (
    SparkSession.builder
    .appName("PokemonIncrementalETL")
    .master("local[*]")
    .getOrCreate()
)

In [6]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, DoubleType
)

In [7]:
schema = StructType([
    StructField("pokemon_id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("height_m", DoubleType(), True),
    StructField("updated_at", StringType(), True),
    StructField("batch_date", StringType(), True)
])

In [8]:
source_df =  spark.read.option("header", True).schema(schema).csv(SOURCE_PATH)

In [9]:
source_df.withColumn(
        "updated_at",
        F.to_timestamp("updated_at", "yyyy-MM-dd HH:mm:ss")
    )

DataFrame[pokemon_id: int, name: string, height_m: double, updated_at: timestamp, batch_date: string]

In [10]:
cutoff = F.to_timestamp(F.lit(RUN_UNTIL))

In [11]:
last_watermark = None

In [12]:
if os.path.exists(WATERMARK_PATH):
    with open(WATERMARK_PATH, "r") as file:
        saved_value = file.read().strip()

    if saved_value:
        last_watermark = saved_value

print("Previously saved watermark:", last_watermark)


Previously saved watermark: None


In [13]:
eligible_df = source_df.filter(F.col("updated_at") <= cutoff)

In [14]:
if last_watermark is not None:
    eligible_df = eligible_df.filter(
        F.col("updated_at") >
        F.to_timestamp(F.lit(last_watermark))
    )

In [15]:
max_timestamp = (
    eligible_df
    .agg(F.max("updated_at").alias("max_timestamp"))
    .first()["max_timestamp"]
)

In [16]:
from pyspark.sql import Window

In [17]:
if max_timestamp is None:
    print("No new records to process.")
else:
    window_spec = (
        Window.partitionBy("pokemon_id")
        .orderBy(F.col("updated_at").desc())
    )

    incoming_df = (
        eligible_df
        .withColumn("row_num", F.row_number().over(window_spec))
        .filter(F.col("row_num") == 1)
        .drop("row_num")
    )
    incoming_df.orderBy("pokemon_id").show(truncate=False)


+----------+----------+--------+-------------------+----------+
|pokemon_id|name      |height_m|updated_at         |batch_date|
+----------+----------+--------+-------------------+----------+
|1         |Bulbasaur |0.7     |2026-10-10 09:00:00|2026-10-10|
|2         |Ivysaur   |1.0     |2026-10-10 09:15:00|2026-10-10|
|3         |Venusaur  |2.0     |2026-10-12 08:00:00|2026-10-12|
|4         |Charmander|0.7     |2026-10-12 08:30:00|2026-10-12|
|5         |Charmeleon|1.2     |2026-10-12 09:00:00|2026-10-12|
|6         |Charizard |1.7     |2026-10-10 10:30:00|2026-10-10|
|7         |Squirtle  |0.5     |2026-10-11 09:00:00|2026-10-11|
|8         |Wartortle |1.0     |2026-10-11 09:15:00|2026-10-11|
|9         |Blastoise |1.6     |2026-10-11 09:30:00|2026-10-11|
|10        |Caterpie  |0.3     |2026-10-11 10:00:00|2026-10-11|
|11        |Metapod   |0.7     |2026-10-12 09:30:00|2026-10-12|
|12        |Butterfree|1.1     |2026-10-12 10:00:00|2026-10-12|
+----------+----------+--------+--------